In [4]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import numpy as np

In [ ]:
df = pd.DataFrame({
    "text": ["विद्यालय", "विध्यालय", "जान्छ", "जान्छ्"],
    "label": [0, 1, 0, 1]
})

In [5]:
# tokenization
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df["text"]).toarray()
y = df["label"].values

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [14]:
import torch
import torch.nn as nn

class WordBiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=64):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            bidirectional=True,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        """
        x: (batch_size, seq_len)
        """

        emb = self.embedding(x)        # (B, T, E)
        lstm_out, _ = self.lstm(emb)   # (B, T, 2H)

        logits = self.fc(lstm_out)     # (B, T, 1)
        probs = self.sigmoid(logits)   # (B, T, 1)

        return probs.squeeze(-1)       # (B, T)

In [16]:
model = WordBiLSTMTagger(vocab_size=len(vectorizer.vocabulary_))
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [18]:
epochs = 10

for epoch in range(epochs):
    model.train()

    optimizer.zero_grad()

    # forward pass
    outputs = model(X_train)  
    # shape: (batch_size, seq_len)

    # compute loss
    loss = criterion(outputs, y_train.float())

    # backprop
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

RuntimeError: Expected tensor for argument #1 'indices' to have one of the following scalar types: Long, Int; but got torch.FloatTensor instead (while checking arguments for embedding)

In [12]:
model.eval()
with torch.no_grad():
    preds = model(X_test).squeeze()
    preds = (preds > 0.5).int()

    accuracy = (preds == y_test.int()).float().mean()
    print("Accuracy:", accuracy.item())

Accuracy: 1.0


In [13]:
def predict(text):
    vec = vectorizer.transform([text]).toarray()
    vec = torch.tensor(vec, dtype=torch.float32)

    with torch.no_grad():
        output = model(vec).item()

    return {
        "error_probability": output,
        "label": int(output > 0.5)
    }